# Compute estimate

Per-step cost of each candidate configuration on the target machine, which decides what fits before the final milestone.

In [1]:
import sys
import time

sys.path.insert(0, "..")

import torch
from torch import nn
from torchvision.models.segmentation import deeplabv3_mobilenet_v3_large, deeplabv3_resnet50

from shared import io as shared_io
from shared.device import count_parameters, describe_hardware, select_device, synchronize
from src.interfaces import IGNORE_INDEX, NUM_CLASSES
from src.utils import load_config, resolve_path

config = load_config(resolve_path("configs/base.yaml"))
split = shared_io.read_json(resolve_path(config["dataset"]["split_file"]))
device = select_device()
batch = config["loader"]["batch_size"]
epochs = config["train"]["epochs"]
steps_per_epoch = -(-split["counts"]["train"] // batch)

print(describe_hardware())
print(f"{split['counts']['train']} training images, batch {batch} -> {steps_per_epoch} steps per epoch, {epochs} epochs")

mps: Apple M4 Pro, 14 cores, 48 GB RAM
2406 training images, batch 8 -> 301 steps per epoch, 40 epochs


## Cost model for the baseline

In [2]:
def block(inputs: int, outputs: int) -> nn.Module:
    return nn.Sequential(
        nn.Conv2d(inputs, outputs, 3, padding=1), nn.BatchNorm2d(outputs), nn.ReLU(inplace=True),
        nn.Conv2d(outputs, outputs, 3, padding=1), nn.BatchNorm2d(outputs), nn.ReLU(inplace=True),
    )


class UNet(nn.Module):
    def __init__(self, base: int = 64, depth: int = 4, classes: int = NUM_CLASSES) -> None:
        super().__init__()
        widths = [base * 2 ** level for level in range(depth + 1)]
        self.down = nn.ModuleList([block(3 if i == 0 else widths[i - 1], widths[i]) for i in range(depth)])
        self.bottom = block(widths[depth - 1], widths[depth])
        self.up = nn.ModuleList([nn.ConvTranspose2d(widths[i + 1], widths[i], 2, stride=2) for i in reversed(range(depth))])
        self.merge = nn.ModuleList([block(widths[i] * 2, widths[i]) for i in reversed(range(depth))])
        self.head = nn.Conv2d(base, classes, 1)
        self.pool = nn.MaxPool2d(2)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        skips = []
        for layer in self.down:
            x = layer(x)
            skips.append(x)
            x = self.pool(x)
        x = self.bottom(x)
        for upsample, merge, skip in zip(self.up, self.merge, reversed(skips)):
            x = merge(torch.cat([upsample(x), skip], dim=1))
        return self.head(x)

## Timing

In [3]:
criterion = nn.CrossEntropyLoss(ignore_index=IGNORE_INDEX)
inference = config["evaluation"]["inference"]


def measure(build, crop: int, freeze: bool = False, warmup: int = 3, timed: int = 5) -> dict:
    """Seconds per training step, and milliseconds per image at inference, for one configuration."""
    model = build().to(device)
    if freeze:
        for parameter in model.backbone.parameters():
            parameter.requires_grad = False
    trainable = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(trainable, lr=config["train"]["lr"])

    def logits(x: torch.Tensor) -> torch.Tensor:
        out = model(x)
        return out["out"] if isinstance(out, dict) else out

    images = torch.randn(batch, 3, crop, crop, device=device)
    targets = torch.randint(0, NUM_CLASSES, (batch, crop, crop), device=device)

    def step() -> None:
        optimizer.zero_grad()
        criterion(logits(images), targets).backward()
        optimizer.step()

    model.train()
    for _ in range(warmup):
        step()
    synchronize(device)
    start = time.perf_counter()
    for _ in range(timed):
        step()
    synchronize(device)
    seconds = (time.perf_counter() - start) / timed

    model.eval()
    latency = {}
    with torch.no_grad():
        for size in inference["batch_sizes"]:
            x = torch.randn(size, 3, crop, crop, device=device)
            for _ in range(inference["warmup_batches"]):
                logits(x)
            synchronize(device)
            start = time.perf_counter()
            for _ in range(inference["timed_batches"]):
                logits(x)
            synchronize(device)
            latency[str(size)] = (time.perf_counter() - start) * 1000 / (inference["timed_batches"] * size)
            del x

    result = {"crop": crop, "trainable": sum(p.numel() for p in trainable), "seconds_per_step": seconds,
              "minutes_per_epoch": seconds * steps_per_epoch / 60,
              "hours_per_run": seconds * steps_per_epoch * epochs / 3600,
              "inference_ms_per_image": latency}
    del model, optimizer, images, targets
    return result


configurations = [
    ("U-Net, base width 32, depth 4", lambda: UNet(base=32), 512, False),
    ("U-Net, base width 32, depth 4", lambda: UNet(base=32), 384, False),
    ("U-Net, base width 64, depth 4", lambda: UNet(base=64), 512, False),
    ("DeepLabV3-MobileNetV3-Large", lambda: deeplabv3_mobilenet_v3_large(weights=None, weights_backbone=None, num_classes=NUM_CLASSES), 512, False),
    ("DeepLabV3-MobileNetV3-Large", lambda: deeplabv3_mobilenet_v3_large(weights=None, weights_backbone=None, num_classes=NUM_CLASSES), 384, False),
    ("DeepLabV3-ResNet50", lambda: deeplabv3_resnet50(weights=None, weights_backbone=None, num_classes=NUM_CLASSES), 512, False),
    ("DeepLabV3-ResNet50, backbone frozen", lambda: deeplabv3_resnet50(weights=None, weights_backbone=None, num_classes=NUM_CLASSES), 512, True),
]

measured = {f"{name} @{crop}": measure(build, crop, freeze) for name, build, crop, freeze in configurations}

sizes = [str(s) for s in inference["batch_sizes"]]
print(f"{'configuration':<40}{'trainable':>11}{'s/step':>9}{'h/run':>8}" + "".join(f"{'ms@' + s:>10}" for s in sizes))
for name, r in measured.items():
    latency = "".join(f"{r['inference_ms_per_image'][s]:>10.1f}" for s in sizes)
    print(f"{name:<40}{r['trainable']/1e6:>10.1f}M{r['seconds_per_step']:>9.2f}{r['hours_per_run']:>8.1f}{latency}")

configuration                             trainable   s/step   h/run      ms@1      ms@8
U-Net, base width 32, depth 4 @512             7.8M     0.73     2.4      22.9      22.5
U-Net, base width 32, depth 4 @384             7.8M     0.41     1.4      13.7      12.5
U-Net, base width 64, depth 4 @512            31.0M     2.05     6.9      70.4      69.6
DeepLabV3-MobileNetV3-Large @512              11.0M     0.25     0.8      14.9       8.1
DeepLabV3-MobileNetV3-Large @384              11.0M     0.15     0.5      11.9       5.4
DeepLabV3-ResNet50 @512                       39.6M     1.73     5.8      73.0      71.6
DeepLabV3-ResNet50, backbone frozen @512      16.1M     0.81     2.7      72.7      71.3


## Planned budget

In [ ]:
plan = {
    "U-Net, 3 seeds": 3 * measured["U-Net, base width 32, depth 4 @512"]["hours_per_run"],
    "DeepLabV3-MobileNetV3, 3 seeds": 3 * measured["DeepLabV3-MobileNetV3-Large @512"]["hours_per_run"],
    "freeze vs fine-tune, 1 seed": measured["DeepLabV3-MobileNetV3-Large @512"]["hours_per_run"],
    "ResNet50 backbone, 1 seed": measured["DeepLabV3-ResNet50 @512"]["hours_per_run"],
}
for name, hours in plan.items():
    print(f"  {name:<34}{hours:>6.1f} h")
print(f"  {'total':<34}{sum(plan.values()):>6.1f} h")

shared_io.write_json(resolve_path("results/compute.json"),
                     {"hardware": describe_hardware(), "batch_size": batch, "epochs": epochs,
                      "steps_per_epoch": steps_per_epoch, "configurations": measured, "planned_budget_hours": plan})

  U-Net, 3 seeds                       7.3 h
  DeepLabV3-MobileNetV3, 3 seeds       2.5 h
  freeze vs fine-tune, 1 seed          0.8 h
  ResNet50 backbone, 1 seed            5.8 h
  total                               16.4 h

wrote results/compute.json
